<a href="https://colab.research.google.com/github/naylanara89/ProyekAkhir_A_Avocado/blob/main/02_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02. Preprocessing Data

**Dataset: Avocado Prices**


## Step 1 — Memuat dataset awal



In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# Sumber dataset asli: Kaggle - Avocado Prices
kaggle_url = 'https://www.kaggle.com/datasets/neuromusic/avocado-prices'

from google.colab import files
uploaded = files.upload()

csv_files = [nama for nama in uploaded.keys() if nama.lower().endswith('.csv')]
if not csv_files:
    raise FileNotFoundError('File CSV belum dipilih. Jalankan ulang sel ini dan pilih avocado.csv.')

nama_file = csv_files[0]
df = pd.read_csv(nama_file)

print('Sumber dataset:', kaggle_url)
print('File yang dibaca:', nama_file)
print('Ukuran data awal:', df.shape)
display(df.head())

Saving avocado.csv to avocado.csv
Sumber dataset: https://www.kaggle.com/datasets/neuromusic/avocado-prices
File yang dibaca: avocado.csv
Ukuran data awal: (18249, 14)


,Unnamed: 0,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
0,0,2015-12-27,1.33,"64,236.62","1,036.74","54,454.85",48.16,"8,696.87","8,603.62",93.25,0.00,conventional,2015,Albany
1,1,2015-12-20,1.35,"54,876.98",674.28,"44,638.81",58.33,"9,505.56","9,408.07",97.49,0.00,conventional,2015,Albany
2,2,2015-12-13,0.93,"118,220.22",794.70,"109,149.67",130.50,"8,145.35","8,042.21",103.14,0.00,conventional,2015,Albany
3,3,2015-12-06,1.08,"78,992.15","1,132.00","71,976.41",72.58,"5,811.16","5,677.40",133.76,0.00,conventional,2015,Albany
4,4,2015-11-29,1.28,"51,039.60",941.48,"43,838.39",75.78,"6,183.95","5,986.26",197.69,0.00,conventional,2015,Albany


## Step 2 — Memeriksa kondisi awal data

In [2]:
print('Informasi dataset:')
df.info()

print('\\nMissing value setiap kolom:')
display(df.isna().sum().to_frame('Jumlah Missing Value'))

print('\\nJumlah baris duplikat:', df.duplicated().sum())

Informasi dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18249 entries, 0 to 18248
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0    18249 non-null  int64  
 1   Date          18249 non-null  object 
 2   AveragePrice  18249 non-null  float64
 3   Total Volume  18249 non-null  float64
 4   4046          18249 non-null  float64
 5   4225          18249 non-null  float64
 6   4770          18249 non-null  float64
 7   Total Bags    18249 non-null  float64
 8   Small Bags    18249 non-null  float64
 9   Large Bags    18249 non-null  float64
 10  XLarge Bags   18249 non-null  float64
 11  type          18249 non-null  object 
 12  year          18249 non-null  int64  
 13  region        18249 non-null  object 
dtypes: float64(9), int64(2), object(3)
memory usage: 1.9+ MB
\nMissing value setiap kolom:


,Jumlah Missing Value
Unnamed: 0,0
Date,0
AveragePrice,0
Total Volume,0
4046,0
4225,0
4770,0
Total Bags,0
Small Bags,0
Large Bags,0


\nJumlah baris duplikat: 0


## Step 3 — Melakukan preprocessing

Langkah pembersihan yang dilakukan:
1. Menghapus kolom indeks `Unnamed: 0` jika tersedia.
2. Mengubah `Date` menjadi tipe datetime.
3. Merapikan penulisan kategori `type` dan `region`.
4. Menghapus 10 wilayah agregat agar analisis berfokus pada wilayah detail.
5. Menghapus `Total Volume` dan `Total Bags` karena keduanya merupakan penjumlahan komponen rinci.
6. Menghapus baris duplikat dan baris yang tidak memiliki tanggal, jenis, atau wilayah.

In [3]:
data = df.copy()

# Menghapus kolom indeks yang bukan fitur analisis
if 'Unnamed: 0' in data.columns:
    data = data.drop(columns=['Unnamed: 0'])

# Merapikan tipe data dan kategori
if 'Date' in data.columns:
    data['Date'] = pd.to_datetime(data['Date'], errors='coerce')

for col in ['type', 'region']:
    if col in data.columns:
        data[col] = data[col].astype('string').str.strip()

if 'type' in data.columns:
    data['type'] = data['type'].str.lower()

# Menghapus wilayah agregat
aggregate_regions = {
    'california', 'greatlakes', 'midsouth', 'northeast',
    'northernnewengland', 'plains', 'southcentral',
    'southeast', 'west', 'totalus'
}
if 'region' in data.columns:
    region_key = data['region'].str.replace(r'\\s+', '', regex=True).str.lower()
    rows_before_region = len(data)
    data = data.loc[~region_key.isin(aggregate_regions)].copy()
    removed_aggregate = rows_before_region - len(data)
else:
    removed_aggregate = 0

# Menghapus kolom total yang merupakan penjumlahan komponen
cols_to_drop = [c for c in ['Total Volume', 'Total Bags'] if c in data.columns]
data = data.drop(columns=cols_to_drop)

# Menghapus duplikat dan data tanpa informasi utama
rows_before_duplicates = len(data)
data = data.drop_duplicates()
removed_duplicates = rows_before_duplicates - len(data)

required_cols = [c for c in ['Date', 'type', 'region'] if c in data.columns]
if required_cols:
    data = data.dropna(subset=required_cols)

# Mengurutkan data agar mudah diperiksa
sort_cols = [c for c in ['Date', 'region', 'type'] if c in data.columns]
if sort_cols:
    data = data.sort_values(sort_cols).reset_index(drop=True)

print('Baris awal:', len(df))
print('Baris setelah preprocessing:', len(data))
print('Kolom akhir:', len(data.columns))
print('Wilayah detail:', data['region'].nunique() if 'region' in data.columns else '-')
print('Wilayah agregat dihapus:', removed_aggregate)
print('Duplikat dihapus:', removed_duplicates)
print('Kolom total yang dihapus:', cols_to_drop)
display(data.head())

Baris awal: 18249
Baris setelah preprocessing: 14869
Kolom akhir: 11
Wilayah detail: 44
Wilayah agregat dihapus: 3380
Duplikat dihapus: 0
Kolom total yang dihapus: ['Total Volume', 'Total Bags']


,Date,AveragePrice,4046,4225,4770,Small Bags,Large Bags,XLarge Bags,type,year,region
0,2015-01-04,1.22,"2,819.50","28,287.42",49.90,"9,186.93",529.53,0.00,conventional,2015,Albany
1,2015-01-04,1.79,57.42,153.88,0.00,"1,162.65",0.00,0.00,organic,2015,Albany
2,2015-01-04,1.00,"364,302.39","23,821.16",82.15,"16,707.15","30,108.64",0.00,conventional,2015,Atlanta
3,2015-01-04,1.76,"1,500.15",938.35,0.00,"1,071.35",336.84,0.00,organic,2015,Atlanta
4,2015-01-04,1.08,"53,987.31","552,906.04","39,995.03","137,146.07","3,990.61",0.00,conventional,2015,BaltimoreWashington


## Step 4 — Mencatat keputusan preprocessing

In [4]:
keputusan = pd.DataFrame([
    ['Unnamed: 0', 'Dihapus jika tersedia', 'Kolom indeks, bukan informasi yang dibutuhkan untuk analisis.'],
    ['Date', 'Diubah menjadi datetime', 'Agar dapat diolah sebagai data waktu.'],
    ['type dan region', 'Dirapikan', 'Menghindari perbedaan kategori akibat spasi atau kapitalisasi.'],
    ['10 wilayah agregat', 'Dihapus', 'Analisis difokuskan pada 44 wilayah detail, bukan data gabungan.'],
    ['Total Volume dan Total Bags', 'Dihapus', 'Keduanya merupakan total dari komponen rinci sehingga dapat menimbulkan multikolinearitas definisional.'],
    ['Baris duplikat', 'Dihapus', 'Menghindari pencatatan yang sama lebih dari sekali.'],
    ['Baris tanpa Date/type/region', 'Dihapus', 'Informasi tersebut diperlukan untuk mengenali waktu, jenis, dan wilayah pengamatan.'],
    ['Encoding dan scaling', 'Belum dilakukan', 'Dilakukan pada tahap pemodelan, setelah pembagian data latih dan data uji untuk mencegah data leakage.']
], columns=['Temuan/Kolom', 'Keputusan', 'Alasan'])

display(keputusan)

,Temuan/Kolom,Keputusan,Alasan
0,Unnamed: 0,Dihapus jika tersedia,"Kolom indeks, bukan informasi yang dibutuhkan ..."
1,Date,Diubah menjadi datetime,Agar dapat diolah sebagai data waktu.
2,type dan region,Dirapikan,Menghindari perbedaan kategori akibat spasi at...
3,10 wilayah agregat,Dihapus,"Analisis difokuskan pada 44 wilayah detail, bu..."
4,Total Volume dan Total Bags,Dihapus,Keduanya merupakan total dari komponen rinci s...
5,Baris duplikat,Dihapus,Menghindari pencatatan yang sama lebih dari se...
6,Baris tanpa Date/type/region,Dihapus,Informasi tersebut diperlukan untuk mengenali ...
7,Encoding dan scaling,Belum dilakukan,"Dilakukan pada tahap pemodelan, setelah pembag..."


## Step 5 — Memeriksa hasil akhir

In [5]:
print('Ukuran data bersih:', data.shape)
print('Missing value setelah preprocessing:')
display(data.isna().sum().to_frame('Jumlah Missing Value'))

print('Jumlah duplikat setelah preprocessing:', data.duplicated().sum())
print('Jumlah wilayah detail:', data['region'].nunique())
print('Distribusi jenis alpukat:')
display(data['type'].value_counts())

Ukuran data bersih: (14869, 11)
Missing value setelah preprocessing:


,Jumlah Missing Value
Date,0
AveragePrice,0
4046,0
4225,0
4770,0
Small Bags,0
Large Bags,0
XLarge Bags,0
type,0
year,0


Jumlah duplikat setelah preprocessing: 0
Jumlah wilayah detail: 44
Distribusi jenis alpukat:


,count
type,
conventional,7436
organic,7433


## Step 6 — Menyimpan dataset bersih

Dataset hasil preprocessing disimpan dengan nama `avocado_bersih.csv`.

In [6]:
output_path = 'avocado_bersih.csv'
data.to_csv(output_path, index=False)

print('Dataset bersih berhasil disimpan:', output_path)

from google.colab import files
files.download(output_path)

Dataset bersih berhasil disimpan: avocado_bersih.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Kesimpulan

Preprocessing dilakukan dengan merapikan tipe data dan kategori, menghapus wilayah agregat, menghilangkan kolom total yang redundan, serta menghapus duplikat dan baris tanpa informasi utama. Encoding, scaling, pembagian train-test, dan pelatihan model belum dilakukan karena termasuk tahap pemodelan berikutnya.